In [0]:
%sql
DROP TABLE db_test.gold.revenue_by_industry;
DROP TABLE db_test.gold.top_movies;
DROP TABLE db_test.gold.revenue_by_year;

DROP TABLE db_test.silver.movies;

DROP TABLE db_test.bronze.movies;

## INGESTION FROM **S3**
Already created an extrenal location in databricks which connects s3 bucket
Now reading the data directly from that location
Adding source, and timecolumn, so that we can create table by parttioning based on date

In [0]:
from pyspark.sql.functions import current_date, col, current_timestamp

df = spark.read.format('csv').option('header','True').load('s3://rjdbtest/raw/movies.csv')
df = df.withColumn("ingestion_date", current_date())\
    .withColumn('Source', col('_metadata.file_name'))\
        .withColumn('ingestion_time',current_timestamp())
    
display(df)

## Loading the extracted data into delta table

In [0]:
df.write.format("delta") \
    .mode("append") \
    .partitionBy("ingestion_date") \
    .saveAsTable("db_test.bronze.movies")

Bronze table created in delta format